In [1]:
from dataset import *
from torch.utils.data import Subset
from torch.utils.data import DataLoader
from torchnet.dataset import TensorDataset
import numpy as np
from autoencoders import *
from train import *
from test import *

In [2]:
######## making dataloaders ##########
def getDataLoaders(datasets, batch_size, shuffle, drop_last):
    datasets = TensorDataset(datasets) 

    dataloader = DataLoader(datasets, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last) #Shuffle here
        
    return dataloader

In [3]:
root_data_path = ""
bulk_mrna_path = root_data_path + "sample_bulk_mrna_data.csv"
bulk_mirna_path = root_data_path + "sample_bulk_mirna_data.csv"
sc_mrna_path = root_data_path + "sample_sc_mrna_data.csv"
sc_mirna_path = root_data_path + "sample_sc_mirna_data.csv"

In [ ]:
######### Loading Dataset ############
print('loading data.......')
bulk_mrna_data = RNA_Dataset(bulk_mrna_path, 'mrna')
bulk_mirna_data = RNA_Dataset(bulk_mirna_path, 'mirna')
sc_mrna_data = RNA_Dataset(sc_mrna_path, 'mrna')
sc_mirna_data = RNA_Dataset(sc_mirna_path, 'mirna')

In [ ]:
print('splitting datasets.......')
train_dataset = [bulk_mrna_data, bulk_mirna_data]
test_dataset = [sc_mrna_data, sc_mirna_data]

print('making dataloaders.......')
train_loader = getDataLoaders(train_dataset, 64, True, True)
test_loader = getDataLoaders(test_dataset, len(test_dataset[0]), False, False)

In [7]:
bs = 64
optimizer = optim.Adam
mirna_features_no = bulk_mirna_data.shape[1]
mrna_features_no = bulk_mrna_data.shape[1]

In [ ]:
for i in range(mirna_features_no):
      print(f"miRNA no: {i+1}")
      input_params = (mrna_features_no, mirna_features_no)
      lr = 0.01
      epochs = 1
      mirna_no = i
      hyper_params = (lr, epochs, bs, optimizer)
  
      print('training model......')
      model  = train(SplicedAutoEncoder, train_loader, test_loader, input_params, hyper_params, mirna_no)
  
      print('Testing Model.......')
      test_pred = test(model, test_loader)
      
      test_pred_df = pd.DataFrame(list(test_pred.squeeze()))
      
      if i == 0:
        final_pred = test_pred_df
      if i > 0:
        final_pred = pd.concat([final_pred, test_pred_df], axis = 1)
      

        
final_pred = final_pred.values
print(final_pred.shape)

for i, batch_features in enumerate(test_loader):
    original = batch_features[1][1].detach().numpy()
    
print(original.shape)